# Implement matmul in 3 ways

In [8]:
import torch

In [9]:
matrix_a_dimension = (3,4)
matrix_b_dimension = (4,5)

matrix_a = torch.randn(matrix_a_dimension)
matrix_b = torch.randn(matrix_b_dimension)
expected_output = matrix_a @ matrix_b
print(expected_output)

tensor([[ 4.0061, -0.1024, -0.1168,  3.3414, -1.1449],
        [ 0.6532, -0.8786,  0.0664,  1.2217, -0.9770],
        [-1.4937, -2.1263,  0.2193,  0.5577, -1.8441]])


In [10]:
# nested loops based matmul
rows_a = matrix_a_dimension[0]
inner_dimension = matrix_a_dimension[1]
column_b = matrix_b_dimension[1]
result = torch.zeros(rows_a, column_b)

for row_a in range(rows_a):
    for col_b in range(column_b):
        for i in range(inner_dimension):
            result[row_a, col_b] += matrix_a[row_a, i] * matrix_b[i, col_b]

print(result)
print(f'\nDoes tensor matmul matches with loop impl: {torch.allclose(expected_output, result)}')

tensor([[ 4.0061, -0.1024, -0.1168,  3.3414, -1.1449],
        [ 0.6532, -0.8786,  0.0664,  1.2217, -0.9770],
        [-1.4937, -2.1263,  0.2193,  0.5577, -1.8441]])

Does tensor matmul matches with loop impl: True


In [11]:
# dot product based matmul
dot_product_result = torch.zeros(rows_a, column_b)
for row_a in range(rows_a):
    for col_b in range(column_b):
        dot_product_result[row_a, col_b] = torch.dot(matrix_a[row_a, :], matrix_b[:, col_b])
print(dot_product_result)
print(f'\nDoes tensor matmul matches with dot product based impl: {torch.allclose(expected_output, dot_product_result)}')

tensor([[ 4.0061, -0.1024, -0.1168,  3.3414, -1.1449],
        [ 0.6532, -0.8786,  0.0664,  1.2217, -0.9770],
        [-1.4937, -2.1263,  0.2193,  0.5577, -1.8441]])

Does tensor matmul matches with dot product based impl: True


In [12]:
# Weighted sum: column col_b of the output = sum over i of matrix_b[i, col_b] * (column i of matrix_a)
weighted_sum_result = torch.zeros(rows_a, column_b)
for col_b in range(column_b):
    for i in range(inner_dimension):
        weighted_sum_result[:, col_b] += matrix_b[i, col_b] * matrix_a[:, i]
print(weighted_sum_result)
print(f'\nDoes tensor matmul matches with weighted sum based impl: {torch.allclose(expected_output, weighted_sum_result)}')

tensor([[ 4.0061, -0.1024, -0.1168,  3.3414, -1.1449],
        [ 0.6532, -0.8786,  0.0664,  1.2217, -0.9770],
        [-1.4937, -2.1263,  0.2193,  0.5577, -1.8441]])

Does tensor matmul matches with weighted sum based impl: True


In [13]:
# Implementing attention computation
batch_size = 4
seq_len = 128
head_dim = 64
Q = torch.randn(batch_size, seq_len, head_dim)
K = torch.randn(batch_size, seq_len, head_dim)

print('Predicted shape of Q @ K.transpose(-2, -1): (4, 128, 128)')
print('Predicted shape of scores / √64 and of their softmax: (4, 128, 128)')

scores = Q @ K.transpose(-2, -1)
print(f'\nActual shape of Q @ K.transpose(-2, -1): {scores.shape}')

scaled_scores = scores / head_dim**0.5
print(f'Actual shape of scores / √{head_dim}: {scaled_scores.shape}')

attention_weights = scaled_scores.softmax(dim=-1)
print(f'Actual shape of softmax: {attention_weights.shape}')

print(f'\nEvery softmax row sums to 1: {torch.allclose(attention_weights.sum(dim=-1), torch.ones(batch_size, seq_len))}')
print(f'Std of scores: {scores.std():.2f} (≈ √{head_dim} = {head_dim**0.5:.0f}), std of scaled scores: {scaled_scores.std():.2f} (≈ 1)')

Predicted shape of Q @ K.transpose(-2, -1): (4, 128, 128)
Predicted shape of scores / √64 and of their softmax: (4, 128, 128)

Actual shape of Q @ K.transpose(-2, -1): torch.Size([4, 128, 128])
Actual shape of scores / √64: torch.Size([4, 128, 128])
Actual shape of softmax: torch.Size([4, 128, 128])

Every softmax row sums to 1: True
Std of scores: 8.04 (≈ √64 = 8), std of scaled scores: 1.01 (≈ 1)


In [14]:
# Check all three implementations against torch.matmul on random shapes
NUMBER_OF_RANDOM_SHAPES = 10
MAX_DIMENSION = 10

for _ in range(NUMBER_OF_RANDOM_SHAPES):
    rows_a, inner_dimension, column_b = torch.randint(1, MAX_DIMENSION + 1, (3,)).tolist()
    matrix_a = torch.randn(rows_a, inner_dimension)
    matrix_b = torch.randn(inner_dimension, column_b)
    expected_output = torch.matmul(matrix_a, matrix_b)

    loop_result = torch.zeros(rows_a, column_b)
    dot_product_result = torch.zeros(rows_a, column_b)
    weighted_sum_result = torch.zeros(rows_a, column_b)
    for row_a in range(rows_a):
        for col_b in range(column_b):
            for i in range(inner_dimension):
                loop_result[row_a, col_b] += matrix_a[row_a, i] * matrix_b[i, col_b]
            dot_product_result[row_a, col_b] = torch.dot(matrix_a[row_a, :], matrix_b[:, col_b])
    for col_b in range(column_b):
        for i in range(inner_dimension):
            weighted_sum_result[:, col_b] += matrix_b[i, col_b] * matrix_a[:, i]

    is_loop_correct = torch.allclose(expected_output, loop_result, atol=1e-5)
    is_dot_product_correct = torch.allclose(expected_output, dot_product_result, atol=1e-5)
    is_weighted_sum_correct = torch.allclose(expected_output, weighted_sum_result, atol=1e-5)
    print(f'({rows_a}, {inner_dimension}) @ ({inner_dimension}, {column_b}): loop={is_loop_correct}, dot={is_dot_product_correct}, weighted_sum={is_weighted_sum_correct}')

(4, 7) @ (7, 7): loop=True, dot=True, weighted_sum=True
(8, 3) @ (3, 3): loop=True, dot=True, weighted_sum=True
(3, 9) @ (9, 2): loop=True, dot=True, weighted_sum=True
(4, 5) @ (5, 7): loop=True, dot=True, weighted_sum=True
(9, 5) @ (5, 4): loop=True, dot=True, weighted_sum=True
(2, 9) @ (9, 7): loop=True, dot=True, weighted_sum=True
(3, 3) @ (3, 10): loop=True, dot=True, weighted_sum=True
(2, 3) @ (3, 8): loop=True, dot=True, weighted_sum=True
(1, 7) @ (7, 1): loop=True, dot=True, weighted_sum=True
(9, 2) @ (2, 5): loop=True, dot=True, weighted_sum=True
